In [0]:
%sql
CREATE VOLUME IF NOT EXISTS delivery.silver.checkpoints;

CREATE TABLE IF NOT EXISTS delivery.silver.order_events (
  event_id STRING, event_type STRING, order_id STRING, customer_id STRING,
  area STRING, amount DOUBLE, status STRING, lat DOUBLE, lon DOUBLE,
  event_time TIMESTAMP, ingested_at TIMESTAMP);

CREATE TABLE IF NOT EXISTS delivery.silver.orders (
  order_id STRING, customer_id STRING, area STRING, amount DOUBLE,
  status STRING, status_rank INT, placed_at TIMESTAMP,
  delivered_at TIMESTAMP, last_event_time TIMESTAMP);

CREATE TABLE IF NOT EXISTS delivery.silver.quarantine (
  raw_payload STRING, reason STRING, bronze_partition INT,
  bronze_offset BIGINT, ingested_at TIMESTAMP);

CREATE TABLE IF NOT EXISTS delivery.silver.dq_results (
  batch_id BIGINT, run_time TIMESTAMP, total_rows BIGINT,
  valid_rows BIGINT, quarantined_rows BIGINT);

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

schema = StructType([
    StructField("event_id", StringType()), StructField("event_type", StringType()),
    StructField("order_id", StringType()), StructField("customer_id", StringType()),
    StructField("area", StringType()), StructField("amount", DoubleType()),
    StructField("status", StringType()), StructField("lat", DoubleType()),
    StructField("lon", DoubleType()), StructField("event_time", StringType()),
])
STATUSES = ["placed", "accepted", "picked_up", "delivered"]
STATUS_ARRAY = "array('placed','accepted','picked_up','delivered')"

def process(batch_df, batch_id):
    if batch_df.isEmpty():
        return
    s = batch_df.sparkSession

    flat = (
        batch_df.withColumn("j", F.from_json("raw_payload", schema))
        .select("raw_payload",
                F.col("partition").alias("bronze_partition"),
                F.col("offset").alias("bronze_offset"),
                "ingested_at", "j.*")
        .withColumn("event_ts", F.try_to_timestamp("event_time"))
    )
    reason = (
        F.when(F.col("event_id").isNull(), "invalid_json")
         .when(F.col("order_id").isNull(), "null_order_id")
         .when(F.col("amount").isNull() | (F.col("amount") <= 0), "bad_amount")
         .when(F.col("status").isNull() | ~F.col("status").isin(STATUSES), "invalid_status")
         .when(F.col("event_ts").isNull(), "bad_event_time")
    )
    flat = flat.withColumn("reason", reason)

    # 1) quarantine bad records
    (flat.filter("reason IS NOT NULL")
         .select("raw_payload", "reason", "bronze_partition", "bronze_offset", "ingested_at")
         .write.mode("append").saveAsTable("delivery.silver.quarantine"))

    # 2) valid events, deduplicated by event_id
    good = (
        flat.filter("reason IS NULL")
        .select("event_id", "event_type", "order_id", "customer_id", "area",
                "amount", "status", "lat", "lon",
                F.col("event_ts").alias("event_time"), "ingested_at")
        .dropDuplicates(["event_id"])
    )
    good.createOrReplaceTempView("v_good")
    s.sql("""MERGE INTO delivery.silver.order_events t USING v_good s
             ON t.event_id = s.event_id
             WHEN NOT MATCHED THEN INSERT *""")

    # 3) one row per order, status only moves forward (handles late events)
    agg = (
        good.withColumn("rank", F.expr(f"array_position({STATUS_ARRAY}, status) - 1"))
        .groupBy("order_id").agg(
            F.first("customer_id").alias("customer_id"),
            F.first("area").alias("area"),
            F.max("amount").alias("amount"),
            F.max("rank").cast("int").alias("status_rank"),
            F.min(F.when(F.col("status") == "placed", F.col("event_time"))).alias("placed_at"),
            F.max(F.when(F.col("status") == "delivered", F.col("event_time"))).alias("delivered_at"),
            F.max("event_time").alias("last_event_time"))
        .withColumn("status", F.expr(f"element_at({STATUS_ARRAY}, status_rank + 1)"))
    )
    agg.createOrReplaceTempView("v_orders")
    s.sql(f"""MERGE INTO delivery.silver.orders t USING v_orders s
              ON t.order_id = s.order_id
              WHEN MATCHED THEN UPDATE SET
                status_rank = greatest(t.status_rank, s.status_rank),
                status = element_at({STATUS_ARRAY}, greatest(t.status_rank, s.status_rank) + 1),
                placed_at = least(t.placed_at, s.placed_at),
                delivered_at = greatest(t.delivered_at, s.delivered_at),
                last_event_time = greatest(t.last_event_time, s.last_event_time)
              WHEN NOT MATCHED THEN INSERT *""")

    # 4) data quality counts
    st = flat.agg(F.count("*").alias("t"),
                  F.sum(F.when(F.col("reason").isNull(), 1).otherwise(0)).alias("v")).first()
    s.sql(f"""INSERT INTO delivery.silver.dq_results
              VALUES ({batch_id}, current_timestamp(), {st['t']}, {st['v']}, {st['t'] - st['v']})""")

In [0]:
(spark.readStream.table("delivery.bronze.orders_raw")
    .writeStream.foreachBatch(process)
    .option("checkpointLocation", "/Volumes/delivery/silver/checkpoints/silver_orders")
    .trigger(availableNow=True)
    .start().awaitTermination())

In [0]:
%sql
SELECT count(*) AS orders, count(DISTINCT order_id) AS distinct_orders FROM delivery.silver.orders;
SELECT count(*) - count(DISTINCT event_id) AS duplicate_events FROM delivery.silver.order_events;
SELECT reason, count(*) AS n FROM delivery.silver.quarantine GROUP BY reason;
SELECT * FROM delivery.silver.dq_results;